# 🌿 Plant Disease Classifier — Production Training Notebook

> **37 classes · EfficientNet-B3 + SE Attention · Colab-ready**  
> Dataset: PlantVillage-origin (~29,629 images) + olive real-world photos  
> Target: Macro-F1 ≥ 0.90 on val | ONNX export for mobile integration

### Notebook Flow
```
A. Environment & Config
B. Dataset & Augmentation
C. Model (EfficientNet-B3 + SE)
D. Loss Functions
E. Training Loop
F. Evaluation & Confusion Matrix
G. GradCAM Visualization
H. ONNX Export
```


---
## A — Environment Setup


In [ ]:
# ── A.1  Install packages (Colab) ────────────────────────────────────────────
import subprocess, sys

pkgs = [
    "timm>=0.9.0",          # EfficientNet-B3 + pretrained weights
    "albumentations>=1.3.0", # augmentation
    "grad-cam>=1.4.6",       # GradCAM visualisation
    "onnx>=1.14.0",          # ONNX export
    "onnxruntime>=1.16.0",   # ONNX validation
    "torchmetrics>=1.0.0",   # Macro-F1, per-class metrics
    "seaborn",
    "tqdm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("✅  Packages ready.")


In [ ]:
# ── A.2  Imports ─────────────────────────────────────────────────────────────
import os, json, time, warnings, random, hashlib
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
from torchvision.utils import make_grid

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchmetrics import F1Score, ConfusionMatrix
from torchmetrics.classification import MulticlassPrecision, MulticlassRecall

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import cv2
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


---
## A.3 — Master Config

> **All hyperparameters in one place.** Change values here only — never hardcode elsewhere.


In [ ]:
# ── A.3  Master Config ───────────────────────────────────────────────────────
CFG = {
    # Paths
    "data_dir"       : "/content/dataset",     # upload dataset here in Colab
    "output_dir"     : "/content/train_output",
    "checkpoint_dir" : "/content/train_output/checkpoints",

    # Data
    "img_size"       : 224,
    "num_classes"    : 37,
    "val_split"      : 0.15,
    "test_split"     : 0.05,
    "seed"           : 42,

    # Model
    "backbone"       : "efficientnet_b3",   # timm model name
    "pretrained"     : True,
    "drop_rate"      : 0.3,                  # dropout before classifier

    # Training
    "epochs"         : 50,
    "batch_size"     : 32,         # reduce to 16 if OOM on Colab T4
    "num_workers"    : 2,

    # Optimiser
    "lr_head"        : 1e-3,
    "lr_backbone"    : 1e-4,
    "weight_decay"   : 1e-4,
    "warmup_epochs"  : 3,          # freeze backbone for first N epochs

    # Scheduler
    "scheduler"      : "cosine",   # cosine annealing
    "eta_min"        : 1e-7,

    # Loss
    "label_smoothing": 0.10,
    "focal_gamma"    : 2.0,        # for olive classes
    "focal_alpha"    : 0.25,

    # Hard / olive classes (from audit)
    "olive_classes"  : ["olive_aculus_olearius","olive_peacock_spot","olive_healthy"],
    "hard_pairs"     : [
        ("Cucumber___Anthracnose","Cucumber___Gummy Stem Blight"),
        ("Bean___angular_leaf_spot","Bean___rust"),
        ("Potato___Late_blight","Potato___healthy"),
        ("Strawberry___Leaf_scorch","Strawberry___healthy"),
        ("Tomato___Early_blight","Tomato___Septoria_leaf_spot"),
    ],

    # Early stopping
    "patience"       : 8,
    "min_delta"      : 0.001,

    # Mixed precision
    "amp"            : True,

    # Augmentation
    "cutmix_alpha"   : 0.4,
    "mixup_alpha"    : 0.2,
    "cutmix_prob"    : 0.4,        # probability of applying CutMix per batch
    "mixup_prob"     : 0.2,

    # Export
    "onnx_opset"     : 17,
}

# Create dirs
for d in [CFG["output_dir"], CFG["checkpoint_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Seed everything
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG["seed"])
print("Config loaded ✅")
print(json.dumps({k:v for k,v in CFG.items() if not isinstance(v,list)}, indent=2))


---
## B — Dataset, Splits & Augmentation


In [ ]:
# ── B.1  Scan dataset & remove exact duplicates ───────────────────────────────
VALID_EXTS = {".jpg",".jpeg",".png",".bmp",".tiff",".webp"}

def md5(path):
    try:
        with open(path,"rb") as f: return hashlib.md5(f.read()).hexdigest()
    except: return None

print("Scanning dataset…")
records = []
for cls_dir in sorted(Path(CFG["data_dir"]).iterdir()):
    if not cls_dir.is_dir(): continue
    for fp in cls_dir.iterdir():
        if fp.suffix.lower() in VALID_EXTS:
            records.append({"class": cls_dir.name, "filepath": str(fp)})

df = pd.DataFrame(records)
print(f"  Raw images   : {len(df):,}")

# Remove exact duplicates (62 found in audit)
print("Computing MD5 hashes for dedup…")
df["md5"] = [md5(fp) for fp in tqdm(df["filepath"], desc="MD5", leave=False)]
before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print(f"  After dedup  : {len(df):,}  (removed {before-len(df)} exact duplicates)")

# Encode labels
classes = sorted(df["class"].unique())
class2idx = {c:i for i,c in enumerate(classes)}
idx2class = {i:c for c,i in class2idx.items()}
df["label"] = df["class"].map(class2idx)

CFG["num_classes"] = len(classes)
CFG["class2idx"]   = class2idx
CFG["idx2class"]   = idx2class
print(f"  Classes      : {len(classes)}")
print("  Classes list :")
for i,c in enumerate(classes):
    print(f"    {i:02d}. {c}")


In [ ]:
# ── B.2  Stratified train / val / test split ──────────────────────────────────
from sklearn.model_selection import StratifiedShuffleSplit

labels = df["label"].values

# First carve out test
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=CFG["test_split"], random_state=CFG["seed"])
train_val_idx, test_idx = next(sss1.split(df["filepath"], labels))

# Then carve val from train_val
train_val_df  = df.iloc[train_val_idx].reset_index(drop=True)
test_df        = df.iloc[test_idx].reset_index(drop=True)
val_ratio_adj  = CFG["val_split"] / (1 - CFG["test_split"])

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio_adj, random_state=CFG["seed"])
tr_idx, vl_idx = next(sss2.split(train_val_df["filepath"], train_val_df["label"]))

train_df = train_val_df.iloc[tr_idx].reset_index(drop=True)
val_df   = train_val_df.iloc[vl_idx].reset_index(drop=True)

print(f"Train : {len(train_df):,}  ({100*len(train_df)/len(df):.1f}%)")
print(f"Val   : {len(val_df):,}   ({100*len(val_df)/len(df):.1f}%)")
print(f"Test  : {len(test_df):,}   ({100*len(test_df)/len(df):.1f}%)")

# Verify no leakage
train_md5s = set(train_df["md5"])
val_md5s   = set(val_df["md5"])
test_md5s  = set(test_df["md5"])
tv_leak = train_md5s & val_md5s
tt_leak = train_md5s & test_md5s
print(f"Train/Val MD5 leaks  : {len(tv_leak)} {'✅' if not tv_leak else '⚠️'}")
print(f"Train/Test MD5 leaks : {len(tt_leak)} {'✅' if not tt_leak else '⚠️'}")


### B.3 — Augmentation Pipelines

Three separate pipelines based on audit findings:
- **Base** — all classes
- **Hard** — confusion-risk pairs (CutMix-friendly heavy augmentation)
- **Olive** — blurry/overlit classes (CLAHE + sharpening)


In [ ]:
# ── B.3  Augmentation pipelines ──────────────────────────────────────────────
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]
S = CFG["img_size"]

# ── Base transform (train) ─────────────────────────────────────────────────
train_base_tf = A.Compose([
    A.RandomResizedCrop(S, S, scale=(0.6, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=30,
                       border_mode=cv2.BORDER_REFLECT, p=0.6),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.8),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3,7)),
        A.MotionBlur(blur_limit=5),
        A.MedianBlur(blur_limit=5),
    ], p=0.3),
    A.GaussNoise(var_limit=(10,50), p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.2),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

# ── Olive transform — CLAHE + sharpen + re-expose ─────────────────────────
train_olive_tf = A.Compose([
    A.RandomResizedCrop(S, S, scale=(0.6, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.7),
    A.Sharpen(alpha=(0.3, 0.7), lightness=(0.7, 1.0), p=0.6),
    A.RandomBrightnessContrast(brightness_limit=(-0.35, 0.1),
                                contrast_limit=(-0.1, 0.3), p=0.7),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03, p=0.6),
    A.GaussNoise(var_limit=(10,40), p=0.3),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

# ── Validation / Test transform — deterministic ────────────────────────────
val_tf = A.Compose([
    A.Resize(S, S),
    A.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ToTensorV2(),
])

OLIVE_SET = set(CFG["olive_classes"])
print("Augmentation pipelines created ✅")


In [ ]:
# ── B.4  PlantDataset class ───────────────────────────────────────────────────
class PlantDataset(Dataset):
    def __init__(self, df, transform_base, transform_olive=None,
                 olive_classes=None, is_train=True):
        self.df             = df.reset_index(drop=True)
        self.tf_base        = transform_base
        self.tf_olive       = transform_olive
        self.olive_classes  = set(olive_classes or [])
        self.is_train       = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["label"])
        cls   = row["class"]

        try:
            img = np.array(Image.open(row["filepath"]).convert("RGB"))
        except Exception:
            img = np.zeros((256, 256, 3), dtype=np.uint8)

        if self.is_train and cls in self.olive_classes and self.tf_olive:
            tensor = self.tf_olive(image=img)["image"]
        else:
            tensor = self.tf_base(image=img)["image"]

        return tensor, label

# ── B.5  Class-balanced WeightedRandomSampler ─────────────────────────────
def make_sampler(train_df):
    class_counts = train_df["label"].value_counts().sort_index().values
    weights_per_class = 1.0 / class_counts
    sample_weights = weights_per_class[train_df["label"].values]
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weights).float(),
        num_samples=len(train_df),
        replacement=True
    )

train_ds = PlantDataset(train_df, train_base_tf, train_olive_tf,
                        CFG["olive_classes"], is_train=True)
val_ds   = PlantDataset(val_df,   val_tf, is_train=False)
test_ds  = PlantDataset(test_df,  val_tf, is_train=False)

sampler  = make_sampler(train_df)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          sampler=sampler,
                          num_workers=CFG["num_workers"],
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"]*2,
                          shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CFG["batch_size"]*2,
                          shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")


In [ ]:
# ── B.6  Visualise one augmented batch ───────────────────────────────────────
def denorm(t, mean=IMG_MEAN, std=IMG_STD):
    m = torch.tensor(mean).view(3,1,1)
    s = torch.tensor(std).view(3,1,1)
    return (t * s + m).clamp(0,1)

imgs, labels = next(iter(train_loader))
grid = make_grid(denorm(imgs[:16]), nrow=8, padding=2)
fig, ax = plt.subplots(figsize=(16,4))
ax.imshow(grid.permute(1,2,0).numpy())
ax.axis("off")
ax.set_title("Augmented training batch (16 samples)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/augmented_batch_preview.png", dpi=120)
plt.show()
print("Classes in this batch:", [idx2class[l.item()] for l in labels[:8]])


---
## C — Model: EfficientNet-B3 + SE Attention Head

**Why EfficientNet-B3 + SE?**  
- EfficientNet-B3 achieves the best accuracy/FLOPs tradeoff for fine-grained texture classification  
- SE (Squeeze-Excitation) in the head forces channel-wise reweighting → attends to disease colour/texture, not background  
- Pretrained on ImageNet → reuses low-level edge/texture detectors, needs little data to fine-tune  


In [ ]:
# ── C.1  Model definition ────────────────────────────────────────────────────
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class PlantDiseaseClassifier(nn.Module):
    def __init__(self, num_classes, backbone="efficientnet_b3",
                 pretrained=True, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained,
            num_classes=0,          # remove default head
            drop_rate=drop_rate,
        )
        feat_dim = self.backbone.num_features
        self.se  = SEBlock(feat_dim, reduction=16)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(feat_dim),
            nn.Dropout(drop_rate),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, num_classes),
        )

    def forward_features(self, x):
        feats = self.backbone.forward_features(x)   # (B, C, H, W)
        feats = self.se(feats)                       # SE attention
        feats = self.pool(feats)                     # (B, C, 1, 1)
        return feats

    def forward(self, x):
        return self.head(self.forward_features(x))


model = PlantDiseaseClassifier(
    num_classes = CFG["num_classes"],
    backbone    = CFG["backbone"],
    pretrained  = CFG["pretrained"],
    drop_rate   = CFG["drop_rate"],
).to(DEVICE)

# Parameter count
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {total/1e6:.2f} M")
print(f"Trainable params : {trainable/1e6:.2f} M")


In [ ]:
# ── C.2  Freeze / unfreeze helpers ───────────────────────────────────────────
def freeze_backbone(model):
    for name, param in model.backbone.named_parameters():
        param.requires_grad = False
    print("Backbone FROZEN — only head trains")

def unfreeze_backbone(model, lr_backbone=None, optimiser=None):
    for name, param in model.backbone.named_parameters():
        param.requires_grad = True
    print("Backbone UNFROZEN — full fine-tuning")
    # If optimiser provided, add backbone params with lower LR
    if optimiser is not None and lr_backbone is not None:
        optimiser.add_param_group({
            "params": [p for p in model.backbone.parameters() if p.requires_grad],
            "lr"    : lr_backbone,
            "name"  : "backbone",
        })

def get_parameter_groups(model):
    """Separate LR for head vs backbone."""
    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.se.parameters()) + list(model.head.parameters())
    return [
        {"params": head_params,     "lr": CFG["lr_head"],     "name": "head"},
        {"params": backbone_params, "lr": CFG["lr_backbone"], "name": "backbone"},
    ]


---
## D — Loss Functions

| Loss | Used for |
|-|-|
| `LabelSmoothingCrossEntropy` | Main training loss — prevents overconfidence |
| `FocalLoss` | Olive classes (blurry/hard) |
| `CutMix / MixUp` | Data-level regularisation for confusion pairs |


In [ ]:
# ── D.1  Label Smoothing Cross-Entropy ───────────────────────────────────────
class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes, smoothing=0.1, reduction="mean"):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
        self.reduction   = reduction

    def forward(self, logits, targets):
        log_prob = F.log_softmax(logits, dim=1)
        # Smooth targets
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_prob)
            smooth_targets.fill_(self.smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        loss = -(smooth_targets * log_prob).sum(dim=1)
        return loss.mean() if self.reduction == "mean" else loss


# ── D.2  Focal Loss ────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, reduction="mean"):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha; self.reduction = reduction

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction="none")
        pt   = torch.exp(-ce)
        loss = self.alpha * (1 - pt) ** self.gamma * ce
        return loss.mean() if self.reduction == "mean" else loss


# ── D.3  CutMix & MixUp helpers ───────────────────────────────────────────
def cutmix_data(x, y, alpha=0.4):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    cx = np.random.randint(W); cy = np.random.randint(H)
    cut_w = int(W * np.sqrt(1 - lam)); cut_h = int(H * np.sqrt(1 - lam))
    x1 = np.clip(cx - cut_w//2, 0, W); x2 = np.clip(cx + cut_w//2, 0, W)
    y1 = np.clip(cy - cut_h//2, 0, H); y2 = np.clip(cy + cut_h//2, 0, H)
    idx = torch.randperm(B, device=x.device)
    x = x.clone(); x[:,:,y1:y2,x1:x2] = x[idx,:,y1:y2,x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return x, y, y[idx], lam

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1-lam) * x[idx]
    return mixed_x, y, y[idx], lam

def mixed_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1-lam) * criterion(logits, y_b)


# ── D.4  Instantiate losses ────────────────────────────────────────────────
criterion_main  = LabelSmoothingCE(CFG["num_classes"], CFG["label_smoothing"]).to(DEVICE)
criterion_focal = FocalLoss(CFG["focal_gamma"], CFG["focal_alpha"]).to(DEVICE)

# Class-index sets for special losses
olive_indices = {class2idx[c] for c in CFG["olive_classes"] if c in class2idx}
print(f"Main criterion  : LabelSmoothingCE (smoothing={CFG['label_smoothing']})")
print(f"Focal criterion : FocalLoss (gamma={CFG['focal_gamma']}) for olive classes {olive_indices}")


---
## E — Training Loop

### Schedule
| Phase | Epochs | Backbone | LR Head | LR Backbone |
|-|-|-|-|-|
| Warmup | 1 – 3 | Frozen | 1e-3 | — |
| Fine-tune | 4 – 50 | Unfrozen | 1e-4 | 1e-5 |
| Hard-class boost | auto (last 10) | Unfrozen | 5e-6 | 5e-6 |


In [ ]:
# ── E.1  Optimiser & Scheduler ───────────────────────────────────────────────
# Start with backbone frozen → only head params needed
freeze_backbone(model)

optimiser = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG["lr_head"], weight_decay=CFG["weight_decay"]
)

# Will be replaced in Phase 2; placeholder for Phase 1
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimiser, T_max=CFG["epochs"], eta_min=CFG["eta_min"]
)

scaler = GradScaler(enabled=CFG["amp"])
print("Optimiser:", optimiser)


In [ ]:
# ── E.2  Metric tracking utilities ───────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.001, mode="max"):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = -np.inf if mode=="max" else np.inf
        self.counter   = 0
        self.stop      = False

    def step(self, value):
        improved = (self.mode=="max" and value > self.best + self.min_delta) or                    (self.mode=="min" and value < self.best - self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return improved


def compute_metrics(all_preds, all_labels, num_classes):
    preds  = torch.tensor(all_preds)
    labels = torch.tensor(all_labels)
    f1_macro = F1Score(task="multiclass", num_classes=num_classes, average="macro")(preds, labels)
    f1_per   = F1Score(task="multiclass", num_classes=num_classes, average="none")(preds, labels)
    prec     = MulticlassPrecision(num_classes=num_classes, average="macro")(preds, labels)
    recall   = MulticlassRecall(num_classes=num_classes, average="macro")(preds, labels)
    acc      = (preds == labels).float().mean()
    return {
        "acc"      : acc.item(),
        "f1_macro" : f1_macro.item(),
        "precision": prec.item(),
        "recall"   : recall.item(),
        "f1_per"   : f1_per.numpy(),
    }


history = {k:[] for k in ["train_loss","val_loss","val_acc","val_f1",
                            "val_prec","val_recall","lr_head"]}
es = EarlyStopping(patience=CFG["patience"], min_delta=CFG["min_delta"], mode="max")
best_f1   = 0.0
best_ckpt = f"{CFG['checkpoint_dir']}/best_model.pt"
print("Metric tracking ready ✅")


In [ ]:
# ── E.3  Single-epoch train / validate functions ──────────────────────────────
def train_one_epoch(model, loader, optimiser, scaler, epoch):
    model.train()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1} train", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        # ── Decide mix strategy ───────────────────────────────────────────
        r = np.random.rand()
        if r < CFG["cutmix_prob"]:
            imgs, y_a, y_b, lam = cutmix_data(imgs, labels, CFG["cutmix_alpha"])
            use_mix = True
        elif r < CFG["cutmix_prob"] + CFG["mixup_prob"]:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG["mixup_alpha"])
            use_mix = True
        else:
            use_mix = False

        optimiser.zero_grad(set_to_none=True)
        with autocast(enabled=CFG["amp"]):
            logits = model(imgs)
            if use_mix:
                loss = mixed_loss(criterion_main, logits, y_a, y_b, lam)
            else:
                # Route olive-class samples through focal loss
                olive_mask = torch.tensor([l.item() in olive_indices for l in labels],
                                          dtype=torch.bool, device=DEVICE)
                if olive_mask.any() and not olive_mask.all():
                    loss_main  = criterion_main( logits[~olive_mask], labels[~olive_mask])
                    loss_olive = criterion_focal(logits[olive_mask],  labels[olive_mask])
                    loss = 0.7*loss_main + 0.3*loss_olive
                elif olive_mask.all():
                    loss = criterion_focal(logits, labels)
                else:
                    loss = criterion_main(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimiser)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimiser); scaler.update()

        total_loss += loss.item() * imgs.size(0)
        n          += imgs.size(0)
        preds       = logits.argmax(1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_preds, all_labels, CFG["num_classes"])
    metrics["loss"] = total_loss / n
    return metrics


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels, all_probs = [], [], []

    for imgs, labels in tqdm(loader, desc="Validating", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast(enabled=CFG["amp"]):
            logits = model(imgs)
            loss   = criterion_main(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        n          += imgs.size(0)
        probs       = F.softmax(logits, dim=1)
        preds       = probs.argmax(1)
        all_probs.extend(probs.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_preds, all_labels, CFG["num_classes"])
    metrics["loss"]   = total_loss / n
    metrics["probs"]  = all_probs
    metrics["preds"]  = all_preds
    metrics["labels"] = all_labels
    return metrics


In [ ]:
# ── E.4  Main training loop ───────────────────────────────────────────────────
print("=" * 60)
print(f"  Training: {CFG['epochs']} epochs | {CFG['num_classes']} classes")
print(f"  Device  : {DEVICE}")
print("=" * 60)

phase2_started = False

for epoch in range(CFG["epochs"]):
    t0 = time.time()

    # ── Phase transition: unfreeze backbone after warmup ─────────────────
    if epoch == CFG["warmup_epochs"] and not phase2_started:
        print(f"\n>>> Phase 2: Unfreezing backbone at epoch {epoch+1}")
        unfreeze_backbone(model)
        optimiser = torch.optim.AdamW(
            get_parameter_groups(model),
            weight_decay=CFG["weight_decay"]
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimiser, T_max=CFG["epochs"] - CFG["warmup_epochs"],
            eta_min=CFG["eta_min"]
        )
        scaler = GradScaler(enabled=CFG["amp"])
        phase2_started = True

    # ── Train ─────────────────────────────────────────────────────────────
    train_m = train_one_epoch(model, train_loader, optimiser, scaler, epoch)
    val_m   = validate(model, val_loader)
    scheduler.step()

    # ── Log ───────────────────────────────────────────────────────────────
    lr_head = optimiser.param_groups[0]["lr"]
    history["train_loss"].append(train_m["loss"])
    history["val_loss"].append(val_m["loss"])
    history["val_acc"].append(val_m["acc"])
    history["val_f1"].append(val_m["f1_macro"])
    history["val_prec"].append(val_m["precision"])
    history["val_recall"].append(val_m["recall"])
    history["lr_head"].append(lr_head)

    dt = time.time() - t0
    print(f"Ep {epoch+1:03d}/{CFG['epochs']} | "
          f"TrLoss={train_m['loss']:.4f} TrAcc={train_m['acc']:.3f} | "
          f"ValLoss={val_m['loss']:.4f} ValAcc={val_m['acc']:.3f} "
          f"MacroF1={val_m['f1_macro']:.4f} | "
          f"LR={lr_head:.2e} | {dt:.0f}s")

    # ── Checkpoint ────────────────────────────────────────────────────────
    improved = es.step(val_m["f1_macro"])
    if improved:
        best_f1 = val_m["f1_macro"]
        torch.save({
            "epoch"      : epoch,
            "state_dict" : model.state_dict(),
            "optimiser"  : optimiser.state_dict(),
            "val_f1"     : best_f1,
            "val_acc"    : val_m["acc"],
            "class2idx"  : class2idx,
            "cfg"        : CFG,
        }, best_ckpt)
        print(f"  ✅  New best Macro-F1={best_f1:.4f} → checkpoint saved")

    if es.stop:
        print(f"\nEarly stopping at epoch {epoch+1}. Best Macro-F1={es.best:.4f}")
        break

print("\n✅  Training complete.")
print(f"   Best Macro-F1 : {best_f1:.4f}")
print(f"   Checkpoint    : {best_ckpt}")


---
## F — Evaluation, Confusion Matrix & Per-Class Report


In [ ]:
# ── F.1  Load best checkpoint & run on test set ──────────────────────────────
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']+1} "
      f"(val Macro-F1={ckpt['val_f1']:.4f})")

test_m = validate(model, test_loader)
print("\n=== TEST SET RESULTS ===")
print(f"  Accuracy : {test_m['acc']:.4f}")
print(f"  Macro-F1 : {test_m['f1_macro']:.4f}")
print(f"  Precision: {test_m['precision']:.4f}")
print(f"  Recall   : {test_m['recall']:.4f}")


In [ ]:
# ── F.2  Training curves ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ep = range(1, len(history["train_loss"])+1)
axes[0].plot(ep, history["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(ep, history["val_loss"],   label="Val Loss",   color="coral")
axes[0].set_title("Loss Curves", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(ep, history["val_acc"],  label="Val Accuracy",  color="green")
axes[1].plot(ep, history["val_f1"],   label="Val Macro-F1",  color="purple")
axes[1].plot(ep, history["val_prec"], label="Val Precision", color="orange", linestyle="--")
axes[1].plot(ep, history["val_recall"],label="Val Recall",   color="brown",  linestyle="--")
axes[1].set_title("Validation Metrics", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(ep, history["lr_head"], color="teal")
axes[2].set_title("Learning Rate (head)", fontweight="bold")
axes[2].set_xlabel("Epoch"); axes[2].set_yscale("log")

plt.suptitle("Training History", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/training_curves.png", dpi=150)
plt.show()


In [ ]:
# ── F.3  Full 37×37 Confusion Matrix ─────────────────────────────────────────
preds_t  = torch.tensor(test_m["preds"])
labels_t = torch.tensor(test_m["labels"])
cm_fn    = ConfusionMatrix(task="multiclass", num_classes=CFG["num_classes"])
cm       = cm_fn(preds_t, labels_t).numpy()

# Normalise by row (true class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(24, 20))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=classes, yticklabels=classes,
            linewidths=0.3, linecolor="white",
            annot_kws={"size": 5}, ax=ax,
            vmin=0, vmax=1)
ax.set_title("Normalised Confusion Matrix (Test Set)", fontsize=14, fontweight="bold")
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True",      fontsize=11)
plt.xticks(rotation=90, fontsize=6)
plt.yticks(rotation=0,  fontsize=6)
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Confusion matrix saved ✅")


In [ ]:
# ── F.4  Per-class F1 bar chart ──────────────────────────────────────────────
f1_per = test_m["f1_per"]
f1_df  = pd.DataFrame({"class": classes, "F1": f1_per}).sort_values("F1")

fig, ax = plt.subplots(figsize=(18, 8))
colors  = plt.cm.RdYlGn(f1_df["F1"].values)
ax.barh(f1_df["class"], f1_df["F1"], color=colors, edgecolor="white", lw=0.4)
ax.axvline(0.9, color="green", linestyle="--", lw=1.5, label="Target F1=0.90")
ax.axvline(f1_df["F1"].mean(), color="orange", linestyle="--", lw=1.5,
           label=f"Mean={f1_df['F1'].mean():.3f}")
for i, (_, row) in enumerate(f1_df.iterrows()):
    ax.text(row["F1"]+0.005, i, f"{row['F1']:.3f}", va="center", fontsize=7.5)
ax.set_title("Per-Class F1 Score (Test Set)", fontsize=14, fontweight="bold")
ax.set_xlabel("F1 Score"); ax.set_xlim(0, 1.05)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n⚠️  Classes BELOW F1=0.80 (need improvement):")
low_f1 = f1_df[f1_df["F1"] < 0.80].sort_values("F1")
print(low_f1.to_string(index=False))


In [ ]:
# ── F.5  Full classification report ──────────────────────────────────────────
from sklearn.metrics import classification_report
report = classification_report(
    test_m["labels"], test_m["preds"],
    target_names=classes, digits=3
)
print(report)

# Save to file
with open(f"{CFG['output_dir']}/classification_report.txt", "w") as f:
    f.write(report)
print("Report saved ✅")


In [ ]:
# ── F.6  Hard-pair confusion analysis ────────────────────────────────────────
print("=== CONFUSION AUDIT — KNOWN HARD PAIRS ===")
print(f"{'Class A':<50} {'Class B':<50} {'A→B':>6} {'B→A':>6}")
print("-"*115)
for cls_a, cls_b in CFG["hard_pairs"]:
    if cls_a not in class2idx or cls_b not in class2idx:
        continue
    ia, ib = class2idx[cls_a], class2idx[cls_b]
    a_to_b = cm[ia, ib]
    b_to_a = cm[ib, ia]
    flag = "⚠️" if (a_to_b + b_to_a) > 10 else "✅"
    print(f"{flag} {cls_a:<48} {cls_b:<48} {a_to_b:>6} {b_to_a:>6}")


---
## G — GradCAM Visualisation

> **Critical trust check.** GradCAM shows WHERE the model looks when making a prediction.  
> We MUST verify it attends to lesion regions — not background/borders.  
> If GradCAM highlights the dark border or uniform background → the model has learned a shortcut.


In [ ]:
# ── G.1  GradCAM setup ───────────────────────────────────────────────────────
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Target the last conv block of EfficientNet-B3
target_layers = [model.backbone.blocks[-1]]   # last feature block

cam = GradCAM(model=model, target_layers=target_layers)
print("GradCAM target layer:", target_layers[0].__class__.__name__)


In [ ]:
# ── G.2  Visualise GradCAM for 3 samples per class ───────────────────────────
def inv_normalise(tensor):
    """Undo ImageNet normalisation → [0,1] numpy."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return (tensor.cpu() * std + mean).clamp(0, 1).permute(1,2,0).numpy()

GRADCAM_DIR = Path(CFG["output_dir"]) / "gradcam"
GRADCAM_DIR.mkdir(exist_ok=True)

# Sample 3 correctly-predicted images per class
sample_idx_by_class = defaultdict(list)
for i, (pred, label) in enumerate(zip(test_m["preds"], test_m["labels"])):
    if pred == label and len(sample_idx_by_class[label]) < 3:
        sample_idx_by_class[label].append(i)

print("Generating GradCAM visualisations…")
for cls_idx, sample_indices in tqdm(sample_idx_by_class.items(), desc="GradCAM classes"):
    cls_name = idx2class[cls_idx]
    fig, axes = plt.subplots(len(sample_indices), 2,
                             figsize=(8, 3*len(sample_indices)))
    if len(sample_indices) == 1:
        axes = [axes]

    for row_i, data_idx in enumerate(sample_indices):
        fp    = test_df.iloc[data_idx]["filepath"]
        img_np = np.array(Image.open(fp).resize((S,S)).convert("RGB")) / 255.0
        img_t  = val_tf(image=(img_np*255).astype(np.uint8))["image"].unsqueeze(0).to(DEVICE)

        grayscale_cam = cam(input_tensor=img_t)[0]
        cam_img = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)

        axes[row_i][0].imshow(img_np); axes[row_i][0].set_title(f"Original", fontsize=8)
        axes[row_i][1].imshow(cam_img); axes[row_i][1].set_title(f"GradCAM", fontsize=8)
        for ax in axes[row_i]: ax.axis("off")

    safe = cls_name.replace("/","_").replace(" ","_")
    plt.suptitle(cls_name, fontsize=9, fontweight="bold")
    plt.tight_layout()
    plt.savefig(GRADCAM_DIR / f"gradcam_{safe}.png", dpi=100, bbox_inches="tight")
    plt.close()

print(f"GradCAM images saved to {GRADCAM_DIR}")
print("⚠️  REVIEW: Open each image — model should highlight the LEAF LESION, not edges/background.")


In [ ]:
# ── G.3  Show GradCAM for known hard-pair classes inline ─────────────────────
hard_classes_flat = [c for pair in CFG["hard_pairs"] for c in pair
                     if c in class2idx][:6]

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
for col_i, cls_name in enumerate(hard_classes_flat):
    cls_idx = class2idx[cls_name]
    idxs    = sample_idx_by_class.get(cls_idx, [])
    if not idxs:
        axes[0,col_i].axis("off"); axes[1,col_i].axis("off"); continue

    fp     = test_df.iloc[idxs[0]]["filepath"]
    img_np = np.array(Image.open(fp).resize((S,S)).convert("RGB")) / 255.0
    img_t  = val_tf(image=(img_np*255).astype(np.uint8))["image"].unsqueeze(0).to(DEVICE)
    gray   = cam(input_tensor=img_t)[0]
    cam_img= show_cam_on_image(img_np.astype(np.float32), gray, use_rgb=True)

    axes[0,col_i].imshow(img_np)
    axes[0,col_i].set_title(cls_name.split("___")[-1], fontsize=7, fontweight="bold")
    axes[0,col_i].axis("off")
    axes[1,col_i].imshow(cam_img)
    axes[1,col_i].set_title("GradCAM", fontsize=7)
    axes[1,col_i].axis("off")

plt.suptitle("GradCAM — Hard Confusion Pairs (verify lesion focus)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/gradcam_hard_pairs.png", dpi=130, bbox_inches="tight")
plt.show()


---
## H — ONNX Export for Mobile Deployment

Exports the model to ONNX format → ready for:
- **Android**: ONNX Runtime Mobile
- **iOS**: CoreML (via `onnx-coreml` or `coremltools`)


In [ ]:
# ── H.1  Export to ONNX ───────────────────────────────────────────────────────
import onnx
import onnxruntime as ort

model.eval()
onnx_path = f"{CFG['output_dir']}/plant_disease_classifier.onnx"

dummy_input = torch.randn(1, 3, S, S, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params     = True,
    opset_version     = CFG["onnx_opset"],
    do_constant_folding= True,
    input_names       = ["input"],
    output_names      = ["output"],
    dynamic_axes      = {"input":  {0: "batch_size"},
                         "output": {0: "batch_size"}},
)
print(f"ONNX model saved: {onnx_path}")

# Validate ONNX graph
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX graph validation ✅")


In [ ]:
# ── H.2  ONNX Runtime inference validation ────────────────────────────────────
sess = ort.InferenceSession(onnx_path,
       providers=["CUDAExecutionProvider","CPUExecutionProvider"])
input_name  = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name

# Run 8 test images and compare with PyTorch output
imgs_val, labels_val = next(iter(val_loader))
imgs_val = imgs_val[:8].to(DEVICE)

with torch.no_grad():
    pt_out = model(imgs_val).cpu().numpy()

ort_out = sess.run([output_name],
                   {input_name: imgs_val.cpu().numpy()})[0]

max_diff = np.abs(pt_out - ort_out).max()
print(f"Max output diff PyTorch vs ONNX: {max_diff:.6f}  {'✅  Match' if max_diff < 1e-4 else '⚠️  Mismatch'}")
print(f"ONNX input  : {input_name}  {sess.get_inputs()[0].shape}")
print(f"ONNX output : {output_name} {sess.get_outputs()[0].shape}")


In [ ]:
# ── H.3  Model info + class index file ───────────────────────────────────────
import os

onnx_size_mb = os.path.getsize(onnx_path) / 1e6
print(f"ONNX model size: {onnx_size_mb:.1f} MB")

# Save class index mapping — needed in mobile app
labels_path = f"{CFG['output_dir']}/class_labels.json"
with open(labels_path, "w") as f:
    json.dump(idx2class, f, indent=2)
print(f"Class labels saved: {labels_path}")

# Final summary
summary = {
    "timestamp"        : datetime.now().isoformat(),
    "backbone"         : CFG["backbone"],
    "num_classes"      : CFG["num_classes"],
    "img_size"         : CFG["img_size"],
    "best_val_f1"      : float(best_f1),
    "test_acc"         : float(test_m["acc"]),
    "test_f1_macro"    : float(test_m["f1_macro"]),
    "test_precision"   : float(test_m["precision"]),
    "test_recall"      : float(test_m["recall"]),
    "onnx_path"        : onnx_path,
    "onnx_size_mb"     : round(onnx_size_mb, 1),
    "preprocessing"    : {
        "resize"       : f"{S}x{S}",
        "normalize_mean": [0.485, 0.456, 0.406],
        "normalize_std" : [0.229, 0.224, 0.225],
    },
    "mobile_inference_note": (
        "Apply same preprocessing in mobile app: resize to 224x224, "
        "divide by 255, subtract mean, divide by std. "
        "Apply softmax to ONNX output. "
        "If max(softmax) < 0.6, show 'uncertain — retake photo'."
    ),
}
with open(f"{CFG['output_dir']}/model_summary.json","w") as f:
    json.dump(summary, f, indent=2)
print("\n=== FINAL MODEL SUMMARY ===")
print(json.dumps(summary, indent=2))


In [ ]:
# ── H.4  Download outputs from Colab ─────────────────────────────────────────
# Run this cell to zip and download everything
import shutil

zip_path = "/content/plant_disease_model_outputs"
shutil.make_archive(zip_path, "zip", CFG["output_dir"])
print(f"Archive created: {zip_path}.zip")

try:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("Download started in browser ✅")
except ImportError:
    print("Not running in Colab — find the zip at:", f"{zip_path}.zip")


---
## ✅ Training Complete

### Files in `train_output/`
```
train_output/
├── best_model.pt                    ← PyTorch checkpoint
├── plant_disease_classifier.onnx    ← Mobile-ready ONNX model
├── class_labels.json                ← idx→class name mapping
├── model_summary.json               ← Performance + export info
├── classification_report.txt        ← Full precision/recall/F1 per class
├── training_curves.png              ← Loss + metric curves
├── confusion_matrix.png             ← 37×37 normalised CM
├── per_class_f1.png                 ← Per-class F1 bar chart
├── augmented_batch_preview.png      ← Augmentation sanity check
├── gradcam_hard_pairs.png           ← GradCAM for confusion pairs
└── gradcam/                         ← GradCAM per class (3 images each)
```

### Mobile App Integration
```
1. Copy plant_disease_classifier.onnx + class_labels.json to your app
2. Preprocess: resize→224×224, /255, subtract [0.485,0.456,0.406],
               divide by [0.229,0.224,0.225]
3. Run ONNX Runtime inference
4. Apply softmax to output logits
5. If max(softmax) < 0.60 → show 'Please retake photo (low confidence)'
6. Else → show class_labels[argmax] + confidence %
```
